# Model 3 on Kaggle

Use this notebook in the configured GPU session with the `m3-kaggle` checkout. Recovery opens your saved 1.6M-step run, loads its checkpoint in a fresh process, checks its action, and displays the evaluations. If moving to a new notebook, save the entire run folder first and copy it from your attached input into the writable checkout's `results/` directory.

For another training experiment, choose `train` and a new output folder in the first cell. Training uses the configured G1 meshes and the repository terrain.

In [ ]:
from pathlib import Path
import json
import os
import statistics
import subprocess
import sys

repo = Path("/kaggle/working/Bipedal-Locomotion")
mode = "recover"
run = repo / "results/m3_1p6m_env8_lr1e4"
if mode not in {"recover", "train"}:
    raise ValueError("Choose recover or train")
if mode == "train" and not (repo / "src/training/m3_pilot.py").is_file():
    raise FileNotFoundError(f"Set repo to your m3-kaggle checkout: {repo}")
env = {**os.environ, "XLA_PYTHON_CLIENT_PREALLOCATE": "false"}

## Prepare verification

Recovery checks the saved policy directly. Training mode updates the checkout’s verifier before launching a fresh run.

In [ ]:
if mode == "train":
    patch = r'''diff --git a/scripts/render_m3_replay.py b/scripts/render_m3_replay.py
index 61df880..5985f80 100644
--- a/scripts/render_m3_replay.py
+++ b/scripts/render_m3_replay.py
@@ -16,6 +16,7 @@ import numpy as np
 from PIL import Image, ImageDraw
 
 from src.training.m3_env import HistoryJoystick
+from src.training.m3_pilot import check_probe_action
 
 
 def rollout(reset, step, policy, seed, horizon):
@@ -109,7 +110,7 @@ def main():
     with np.load(run / "reload_probe.npz") as probe:
         obs = {key: probe[key] for key in settings["observation_shapes"]}
         action, _ = trained_policy(obs, jax.random.PRNGKey(0))
-        np.testing.assert_allclose(action, probe["expected_action"], atol=1e-6, rtol=1e-6)
+        check_probe_action(action, probe["expected_action"])
 
     config = registry.get_default_config(settings["task"])
     config.naconmax = settings["warp_contact_capacity"]
diff --git a/src/training/m3_pilot.py b/src/training/m3_pilot.py
index 490a3de..256e9f4 100644
--- a/src/training/m3_pilot.py
+++ b/src/training/m3_pilot.py
@@ -124,6 +124,15 @@ def verify_physics(env):
     raise ValueError("G1 foot sensors detected no contact with the generated floor")
 
 
+def check_probe_action(actual, expected):
+    """Compare bounded float32 actions across training and inference processes."""
+    actual, expected = np.asarray(actual), np.asarray(expected)
+    if actual.shape != expected.shape or not (np.isfinite(actual).all() and np.isfinite(expected).all()):
+        raise ValueError("Checkpoint probe actions must have matching shapes and finite values")
+    np.testing.assert_allclose(actual, expected, atol=5e-6, rtol=1e-6)
+    return float(np.max(np.abs(actual - expected)))
+
+
 def verify_checkpoint(directory):
     path = Path(directory).resolve()
     checkpoints = [p for p in (path / "checkpoints").iterdir() if p.name.isdigit()]
@@ -138,8 +147,8 @@ def verify_checkpoint(directory):
         )
         policy = networks.make_inference_fn(network)(checkpoint.load(latest), deterministic=True)
         actual, _ = policy(obs, jax.random.PRNGKey(0))
-        np.testing.assert_allclose(actual, probe["expected_action"], atol=1e-6, rtol=1e-6)
-    save_json(path / "reload.json", {"checkpoint": latest.name, "action_verified": True})
+        max_error = check_probe_action(actual, probe["expected_action"])
+    save_json(path / "reload.json", {"checkpoint": latest.name, "action_verified": True, "max_abs_error": max_error})
 
 
 def run(args):
'''
    def git_apply(*flags):
        return subprocess.run(["git", "apply", *flags, "-"], cwd=repo, input=patch,
                              text=True, capture_output=True)

    if git_apply("--reverse", "--check").returncode == 0:
        print("Action check is already updated.")
    else:
        result = git_apply("--check")
        if result.returncode:
            raise RuntimeError(result.stderr)
        git_apply().check_returncode()
        print("Updated checkpoint and replay action checks.")

## Recover or train

In [ ]:
if mode == "recover":
    for name in ("settings.json", "training.json", "reload_probe.npz", "checkpoints"):
        if not (run / name).exists():
            raise FileNotFoundError(f"Missing {run / name}. Check the saved run folder.")
    verifier = """import json
import sys
from pathlib import Path

import jax
import numpy as np
from brax.training.acme import running_statistics
from brax.training.agents.ppo import checkpoint, networks

run = Path(sys.argv[1]).resolve()
settings = json.loads((run / "settings.json").read_text())
training = json.loads((run / "training.json").read_text())
step = training["environment_steps"]
checkpoints = [p for p in (run / "checkpoints").iterdir()
               if p.is_dir() and p.name.isdigit() and int(p.name) == step]
if len(checkpoints) != 1:
    raise FileNotFoundError(f"Expected one completed checkpoint for step {step} in {run}")
with np.load(run / "reload_probe.npz") as probe:
    obs = {k: probe[k] for k in settings["observation_shapes"]}
    preprocess = running_statistics.normalize if settings["ppo"]["normalize_observations"] else lambda x, _: x
    network = networks.make_ppo_networks({k: v.shape for k, v in obs.items()}, settings["actions"],
                                         preprocess_observations_fn=preprocess, **settings["network"])
    policy = networks.make_inference_fn(network)(checkpoint.load(checkpoints[0]), deterministic=True)
    actual = np.asarray(policy(obs, jax.random.PRNGKey(0))[0])
    expected = probe["expected_action"]
    if actual.shape != expected.shape or not (np.isfinite(actual).all() and np.isfinite(expected).all()):
        raise ValueError("Invalid action shape or non-finite probe values")
    np.testing.assert_allclose(actual, expected, atol=5e-6, rtol=1e-6)
    error = float(np.max(np.abs(actual - expected)))
report = dict(checkpoint=checkpoints[0].name, action_verified=True, max_abs_error=error)
(run / "reload.json").write_text(json.dumps(report, indent=2) + chr(10))
"""
    command = [sys.executable, "-u", "-c", verifier, str(run)]
else:
    if run.exists() and any(run.iterdir()):
        raise ValueError("Choose a new run folder for training.")
    command = [sys.executable, "-u", "scripts/train_kaggle.py", "--model", "M3",
               "--seed", "42", "--num-envs", "8", "--num-timesteps", "1600000",
               "--output-dir", str(run)]
result = subprocess.run(command, cwd=run if mode == "recover" else repo, env=env,
                        stderr=subprocess.PIPE, text=True)
if result.returncode:
    raise RuntimeError(result.stderr[-8000:] or f"Process exited with code {result.returncode}")
print((run / "reload.json").read_text())

## Compare the initial and trained policies

In [ ]:
settings = json.loads((run / "settings.json").read_text())
print("Learning rate:", settings["ppo"]["learning_rate"])
for stage in ("initial", "trained"):
    episodes = json.loads((run / f"{stage}_evaluation.json").read_text())["episodes"]
    duration = statistics.mean(e["steps"] for e in episodes) * settings["control_dt"]
    error = statistics.mean(e["tracking_error_mps"] for e in episodes)
    reward = statistics.mean(e["episode_return"] for e in episodes)
    ended = sum(e["fall"] for e in episodes)
    print(f"{stage}: {duration:.3f} s, tracking error {error:.3f} m/s, "
          f"return {reward:.3f}, {ended}/{len(episodes)} early terminations")